# Fase 1 — Preprocesamiento y Aplanado

Transforma la jerarquía `sessions → levels → rooms` exportada de Firestore
en tres DataFrames planos listos para análisis, y genera las variables derivadas
necesarias para el EDA y el modelo predictivo.

> **Nota sobre el volumen de datos:** Los datos actuales son una muestra inicial
> de desarrollo (11 sesiones). El pipeline está diseñado para escalar automáticamente
> cuando se recojan más sesiones — basta con re-ejecutar `01_extraction.ipynb` y este notebook.

**Entradas:** `data/raw/sessions_raw.json`  
**Salidas:** `data/processed/sessions.parquet`, `levels.parquet`, `rooms.parquet`

## 0. Configuración

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))
from preprocessing import run_pipeline, rooms_to_long

RAW_PATH       = Path('..') / 'data' / 'raw'       / 'sessions_raw.json'
PROCESSED_DIR  = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Raw:       {RAW_PATH.resolve()}')
print(f'Processed: {PROCESSED_DIR.resolve()}')

## 1. Aplanado de la jerarquía

In [ ]:
df_sessions, df_levels, df_rooms = run_pipeline(str(RAW_PATH))

print(f'df_sessions : {df_sessions.shape[0]} filas × {df_sessions.shape[1]} columnas')
print(f'df_levels   : {df_levels.shape[0]} filas × {df_levels.shape[1]} columnas')
print(f'df_rooms    : {df_rooms.shape[0]} filas × {df_rooms.shape[1]} columnas')

In [ ]:
# Vista general de sesiones
df_sessions[['sessionId','playerElement','platform','gameVersion',
             'isVictory','totalTimeSecs','totalKills','totalDeaths',
             'levelsCompleted','hasComment']]

## 2. Limpieza — TFM-11

Identificamos y tratamos sesiones problemáticas antes del análisis.

In [ ]:
print('=== Valores nulos por columna (sesiones) ===')
nulls = df_sessions.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else 'Sin valores nulos')

print('\n=== Tipos de datos ===')
print(df_sessions.dtypes)

In [ ]:
# Detectar sesiones sospechosas de ser pruebas o demasiado cortas
MIN_TIME_SECS = 30   # sesiones de menos de 30s probablemente son pruebas
MAX_TIME_SECS = 3600 # sesiones de más de 1 hora son outliers claros

mask_short   = df_sessions['totalTimeSecs'] < MIN_TIME_SECS
mask_long    = df_sessions['totalTimeSecs'] > MAX_TIME_SECS
mask_nokills = (df_sessions['totalKills'] == 0) & (df_sessions['levelsCompleted'] == 0)

print(f'Sesiones muy cortas  (<{MIN_TIME_SECS}s): {mask_short.sum()}')
print(df_sessions[mask_short][['sessionId','totalTimeSecs','totalKills','playerElement']])

print(f'\nSesiones muy largas  (>{MAX_TIME_SECS}s): {mask_long.sum()}')
print(df_sessions[mask_long][['sessionId','totalTimeSecs','totalKills','playerElement']])

print(f'\nSesiones sin kills ni niveles completados: {mask_nokills.sum()}')
print(df_sessions[mask_nokills][['sessionId','totalTimeSecs','totalKills','levelsCompleted']])

In [ ]:
# Detectar niveles incompletos
niveles_incompletos = df_levels[df_levels['incomplete'] == True]
print(f'Niveles incompletos: {len(niveles_incompletos)}')
print(niveles_incompletos[['sessionId','levelId','kills','deaths','timeSecs']])

In [ ]:
# Decisión de limpieza:
# - Conservamos TODAS las sesiones aunque sean cortas o tengan 0 kills.
#   Con pocos datos no podemos permitirnos descartar registros.
# - Marcamos las sospechosas con un flag para poder filtrarlas en análisis específicos.
# - El outlier de tiempo (3647s) se mantiene — puede ser una sesión AFK real.

df_sessions['is_suspicious'] = mask_short | mask_long | mask_nokills

print(f'Sesiones totales    : {len(df_sessions)}')
print(f'Sesiones sospechosas: {df_sessions["is_suspicious"].sum()}')
print(f'Sesiones limpias    : {(~df_sessions["is_suspicious"]).sum()}')

> **Criterio:** Con el volumen actual (11 sesiones) no descartamos ninguna.
> El flag `is_suspicious` permite filtrar en análisis que requieran datos de calidad.

## 3. Variables calculadas — TFM-12

In [ ]:
# Variables derivadas de sesión
cols_features = ['sessionId','playerElement','isVictory',
                 'kd_ratio','kills_per_min','time_per_level',
                 'completion_rate','totalRooms','totalCast']

print('=== Features de sesión ===')
print(df_sessions[cols_features].to_string())

In [ ]:
# Variables derivadas de sala
cols_room_features = ['sessionId','levelId','roomId',
                      'total_kills_room','total_cast_room','total_miss_room',
                      'cast_accuracy','kills_per_sec','deaths','damageTaken']

print('=== Features de sala (primeras 10) ===')
print(df_rooms[cols_room_features].head(10).to_string())

In [ ]:
# Inventario de hechizos y enemigos detectados
spell_types  = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('cast_')})
enemy_types  = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('kills_')})
status_types = sorted({c.split('_',1)[1] for c in df_rooms.columns if c.startswith('status_')})

print(f'Hechizos detectados ({len(spell_types)}): {spell_types}')
print(f'Enemigos detectados ({len(enemy_types)}): {enemy_types}')
print(f'Efectos de estado  ({len(status_types)}): {status_types}')

In [ ]:
# Formato long para hechizos (útil para comparar uso entre tipos)
df_cast_long = rooms_to_long(df_rooms, 'cast')
print(f'Lanzamientos por hechizo (formato long): {len(df_cast_long)} filas')
print(df_cast_long.groupby('cast_type')['value'].sum().sort_values(ascending=False))

In [ ]:
# Formato long para kills por enemigo
df_kills_long = rooms_to_long(df_rooms, 'kills')
print(f'Kills por enemigo (formato long): {len(df_kills_long)} filas')
print(df_kills_long.groupby('kills_type')['value'].sum().sort_values(ascending=False))

## 4. Estadísticas descriptivas rápidas

In [ ]:
print('=== Sesiones por elemento ===')
print(df_sessions.groupby('playerElement').agg(
    sesiones=('sessionId','count'),
    victorias=('isVictory','sum'),
    kills_media=('totalKills','mean'),
    tiempo_medio=('totalTimeSecs','mean')
).round(1))

In [ ]:
print('=== Dificultad por nivel (deaths + damageTaken medios) ===')
print(df_levels.groupby('levelId').agg(
    sesiones=('sessionId','count'),
    muertes_media=('deaths','mean'),
    daño_media=('damageTaken','mean'),
    tiempo_medio=('timeSecs','mean'),
    intentos_medio=('attempt','mean')
).round(1).sort_values('levelId'))

## 5. Guardar en data/processed/

In [ ]:
# Guardar como Parquet (más eficiente que CSV para datos con muchas columnas)
df_sessions.to_parquet(PROCESSED_DIR / 'sessions.parquet', index=False)
df_levels.to_parquet(  PROCESSED_DIR / 'levels.parquet',   index=False)
df_rooms.to_parquet(   PROCESSED_DIR / 'rooms.parquet',    index=False)

# También CSV para inspección manual
df_sessions.to_csv(PROCESSED_DIR / 'sessions.csv', index=False)
df_levels.to_csv(  PROCESSED_DIR / 'levels.csv',   index=False)
df_rooms.to_csv(   PROCESSED_DIR / 'rooms.csv',    index=False)

print('✓ Archivos guardados en data/processed/')
for f in sorted((PROCESSED_DIR).iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

## 6. Conclusiones

- **Datos actuales:** 11 sesiones, 18 niveles, 82 salas — muestra de desarrollo, insuficiente para modelos estadísticos robustos. Se actualizará cuando haya más testers.
- **Limpieza:** No se descarta ninguna sesión. Flag `is_suspicious` para las 4 sesiones muy cortas o outliers de tiempo.
- **Columnas derivadas creadas:** `kd_ratio`, `kills_per_min`, `time_per_level`, `completion_rate`, `totalRooms`, `totalCast`, `cast_accuracy`, `kills_per_sec`.
- **Hechizos:** Projectile y Blast son los más usados; AOE y Beam aparecen puntualmente.
- **Enemigos:** Variedad de 15+ tipos; los boss tienen kills bajas (esperado).

**Siguiente paso:** `03_eda.ipynb` — visualizaciones exploratorias.